# Emissions-driven models of atmospheric CO₂
**Week 11 · Notebook 2 of 2 · November 19, 2026**

In the first notebook, we fitted functions of time to the Keeling curve. How can we ask what happens if emissions change?

Start with a carbon budget that has the same structure as our earlier EBM. Compare its concentration curve with observations and with quadratic-plus-sine, then add more detail to the carbon response. We will give both dynamical models the same emissions experiments.

All code is supplied. **Make a guess or work it out, run, then explain.** Write your responses at the prediction and discussion prompts. This notebook runs independently of Notebook 1.

Download just this notebook and run the cells in order in the course Python environment. The first cell retrieves the fixed data and supplied model helper from our course GitHub repository. No separate data folder or helper download is needed. The first run needs internet access; saved files can be reused offline. References are at the end.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

COURSE_DATA = 'https://raw.githubusercontent.com/dhruvbalwada/intro-climate-modeling-fall2026/f6e9334448fd452280d2d7fdd913762e7447fce9/labs/data/keeling/'

def course_file(local_name, remote_name, expected_sha256):
    """Download a fixed course file once; reuse the verified local copy."""
    path = Path(local_name)
    if not path.exists():
        print("Downloading", path.name, "from the course GitHub repository...")
        with urlopen(COURSE_DATA + remote_name, timeout=45) as response:
            content = response.read()
        if hashlib.sha256(content).hexdigest() != expected_sha256:
            raise ValueError("Downloaded file differs from the fixed classroom version.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f"{path} differs from the classroom version. Rename it and rerun to download a fresh copy.")
    return path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

course_file('data/mauna_loa_monthly_1958-2025.csv', 'mauna_loa_monthly_1958-2025.csv', 'd29ae206a7f59ae0ffedf2dbcda65b43d1a440bc2628fc19bdb394d76d61cf3e')
course_file('data/historical_budget_1850-2024.csv', 'historical_budget_1850-2024.csv', '317c5ad5f68e4b724e608f124dbfd06c5d033c48e07cbbc4a32acbf97d7bf167')
course_file('data/co2_provenance.json', 'co2_provenance.json', '827868a02c54780340e8ea4a5e88ed86048c9cccf7e1010fae11d200b679b137')
course_file('data/emissions_provenance.json', 'emissions_provenance.json', '9e85586be46398964128d576f2e4254f6109ca0e82bd7fc3f3f430716e0c2d42')
course_file('carbon_models.py', 'carbon_models.py', '1968e394bf32f5075a2c21b79f1f7b9389984486f6d6afafc0cd0f4663db2c91')
from carbon_models import fit_one_box, one_box_trend, fit_four_box, four_box_trend

## 1. Concentration observations and an independent emissions input

Use the same fixed Scripps Mauna Loa record as Notebook 1. These are station observations; the emissions input is global, so our carbon boxes approximate the large-scale trend. The emissions data come from the **Global Carbon Budget 2025**, covering fossil/industrial emissions and land-use change, with the separately reported cement-carbonation sink subtracted.

The emissions record ends in **2024**. All comparisons here use concentration observations through 2024: fit **1958–2016**, evaluate **2017–2024**. Notebook 1 also evaluates 2025, so its test score is slightly different.

Emissions in GtC/year are converted once to a concentration-equivalent rate:

$$e(t)=E(t)/2.124\quad\text{in ppm/year}.$$

This is the increase that emissions would cause **before natural uptake**. It is not the observed increase in the Keeling curve. For the historical calculation we hold each annual emissions rate constant within that calendar year.

In [ ]:
co2 = pd.read_csv(Path("data/mauna_loa_monthly_1958-2025.csv"))
co2 = co2.loc[co2.date < 2025].copy()
budget = pd.read_csv(Path("data/historical_budget_1850-2024.csv"))
years_all = budget["Year"].to_numpy(dtype=int)
E_all = (budget["fossil emissions excluding carbonation"]
         + budget["land-use change emissions"]
         - budget["cement carbonation sink"].fillna(0)).to_numpy()
e_all = E_all / 2.124

T, C = co2.date.to_numpy(), co2.co2.to_numpy()
training = T < 2017
later = ~training
recent = years_all >= 1958
years, e = years_all[recent], e_all[recent]
print(f"Fit: {training.sum()} monthly observations before 2017")
print(f"Evaluate: {later.sum()} monthly observations in 2017–2024")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), layout="constrained")
axes[0].plot(T, C, color="0.35", lw=1)
axes[0].set(ylabel="Mauna Loa CO₂ (ppm)", title="Observed concentration")
axes[1].step(years, e, where="post", color="#B54C2F")
axes[1].set(xlabel="Year", ylabel="Emissions (ppm/year)", title="Supplied global emissions before natural uptake")
for ax in axes: ax.axvspan(2017, 2025, color="0.92", zorder=-1)
plt.show()

## 2. Keep quadratic-plus-sine as the reference

Refit the same four-parameter function from Notebook 1, using only the training observations:

$$\widehat C(t)=a+b u+c u^2+d\sin(2\pi u),\qquad u=t-1958.$$

We retain the same fixed-phase annual sine for every model in this notebook. The quadratic model receives calendar time as input.

In [ ]:
u = T - 1958
season = np.sin(2*np.pi*u)
X = np.column_stack([np.ones(len(T)), u, u**2, season])
quad_parameters = np.linalg.lstsq(X[training], C[training], rcond=None)[0]
quad_prediction = X @ quad_parameters
quad_trend = X[:, :3] @ quad_parameters[:3]
print("Quadratic-plus-sine fitted on training years only.")

## 3. A familiar dynamical equation: one carbon box

As in the EBM, describe a departure from a reference state. Let $x=C_b-C_{\mathrm{ref}}$, where $C_b$ is the slowly varying concentration and $C_{\mathrm{ref}}=280$ ppm is fixed for this toy model:

$$\boxed{\frac{dx}{dt}=e(t)-\frac{x}{\tau}.}$$

Recall the forcing–feedback model from class:

$$C_s\frac{dT_s'}{dt}=F+\alpha T_s',\qquad \alpha<0.$$

$C_s$ is the surface heat capacity per unit area, and $T_s'$ is the surface temperature anomaly. Both equations describe a departure from a reference state. An input adds to a stored quantity, while a restoring term opposes the departure. In the carbon model, $-x/\tau$ represents net uptake by land and ocean.

| Quantity | Meaning | Units |
|---|---|---|
| $x$ | Excess background CO₂ above the reference | ppm |
| $e(t)$ | Emissions before natural uptake | ppm/year |
| $x/\tau$ | Net uptake in this model | ppm/year |
| $\tau$ | Effective uptake timescale | years |

A larger $\tau$ gives slower uptake for the same excess concentration. The model's background concentration is $C_b=280+x$.

Add the seasonal cycle to the model's output:

$$\widehat C(t)=280+x(t)+A\sin(2\pi(t-1958)).$$

The supplied routine jointly fits **$\tau$, the initial excess at the start of 1958, and $A$**, using only the training period. It integrates the equation exactly for the constant emissions rate within each year.

In [ ]:
box = fit_one_box(T, C, training, years, e, reference=280.0)
print(f"Fitted effective uptake timescale: {box['tau']:.1f} years")
print(f"Fitted initial excess at the start of 1958: {box['initial_excess']:.1f} ppm")
print(f"Seasonal sine amplitude: {box['amplitude']:.2f} ppm")

<a id="zero-emissions"></a>

**Guess before running:** If emissions suddenly become zero while excess concentration is positive, does concentration stop changing immediately? What does this equation predict?

**Answer:** [Your answer here.]

### Check the fit before using the model

The concentration observations after 2016 do not enter either parameter fit. Historical emissions are supplied throughout the comparison, including the later years: these are concentration predictions conditional on those inputs, not forecasts of emissions. The later period has already been examined while developing the examples, so this is an exploratory retrospective comparison.

In [ ]:
def error_row(name, prediction):
    return {"Model": name,
            "Training RMSE (ppm)": np.sqrt(np.mean((prediction[training]-C[training])**2)),
            "2017–2024 RMSE (ppm)": np.sqrt(np.mean((prediction[later]-C[later])**2))}

initial_scores = pd.DataFrame([error_row("Quadratic + sine", quad_prediction),
                               error_row("One box + sine", box["prediction"])]).set_index("Model")
display(initial_scores.round(2))

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), layout="constrained")
axes[0].plot(T, C, color="0.6", lw=1, label="Observations")
for pred, label, color in [(quad_prediction, "Quadratic + sine", "#2673A1"),
                           (box["prediction"], "One box + sine", "#B54C2F")]:
    axes[0].plot(T, pred, color=color, lw=1, label=label)
    axes[1].plot(T[later], (pred-C)[later], color=color, label=label)
axes[0].axvspan(2017, 2025, color="0.92", zorder=-1)
axes[0].set(ylabel="CO₂ (ppm)", title="Historical concentration comparison")
axes[0].legend(fontsize=9)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set(xlabel="Year", ylabel="Prediction − observed (ppm)", title="Later-period errors")
axes[1].legend(fontsize=9)
plt.show()

<a id="fit-judgment"></a>

**Discuss:** Which model reproduces the later observations better? What feature of the one-box results suggests its trend response needs improvement?

**Answer:** [Your answer here.]

## 4. Change the emissions input

Keep all fitted parameters fixed. Starting in **2025**, just after the historical emissions record ends, prescribe three illustrative pathways: hold emissions at their 2024 rate, reduce them to zero by 2050, or set them to zero immediately.

These are counterfactual experiments starting at the end of our data, not statements about actual emissions after 2024. We omit the seasonal wiggle from the future panels to make the slowly varying responses easier to compare.

The quadratic continuation is identical for every pathway because that model has no emissions input.

In [ ]:
future_years = np.arange(2025, 2076)
future_times = 2025 + np.arange(12*50 + 1)/12
last_rate = e[-1]
pathways = {
    "Constant emissions": np.full(len(future_years), last_rate),
    "Decline to zero by 2050": last_rate*np.clip((2050-future_years)/25, 0, 1),
    "Zero emissions from 2025": np.zeros(len(future_years)),
}
all_years = np.r_[years_all, future_years]
recent_years = np.r_[years, future_years]
future_u = future_times-1958
quad_future = (quad_parameters[0] + quad_parameters[1]*future_u
               + quad_parameters[2]*future_u**2)
colors = ["#B54C2F", "#27816A", "#775A9E"]
box_future = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for (name, rates), color in zip(pathways.items(), colors):
    box_future[name] = one_box_trend(future_times, box, recent_years, np.r_[e, rates])
    axes[0].step(future_years, rates, where="post", color=color, label=name)
    axes[1].plot(future_times, box_future[name], color=color, label=name)
axes[1].plot(future_times, quad_future, color="0.4", ls="--", label="Quadratic continuation")
axes[0].set(xlabel="Year", ylabel="Emissions (ppm/year)", title="Prescribed inputs")
axes[1].set(xlabel="Year", ylabel="CO₂ trend (ppm)", title="One-box responses")
for ax in axes: ax.legend(fontsize=8)
plt.show()

<a id="scenario-interpretation"></a>

**Discuss:** Why can concentration continue rising under constant emissions? Under zero emissions, where does this model eventually take the concentration? What assumption controls that outcome?

**Answer:** [Your answer here.]

## 5. A richer carbon response

Land and ocean uptake operate over different timescales. A simplified **FaIR-style model** represents several responses: relatively fast uptake, slower adjustments, and a very persistent atmospheric remainder. Its uptake strength also changes with the evolving carbon cycle.

The response rule is guided by calibration against more comprehensive carbon-cycle models. **For this classroom adaptation, the uptake-rule coefficients are refitted to the training concentration observations.** Published partition fractions and base timescales remain fixed; temperature feedback is omitted. We retain the same fitted annual sine as before.

Earlier emissions, from 1850 onward, initialize these response components before the Mauna Loa record begins. The prototype starts at 285 ppm in 1850 with zero additional component anomalies; this is an approximate initialization. The one-box model instead fits its initial 1958 excess.

We will compare the historical concentration curves, then repeat the same emissions experiments. Focus on what changes when the model has more than one adjustment timescale and uptake strength can evolve.

In [ ]:
richer = fit_four_box(T, C, training, years_all, e_all)
predictions = {"Quadratic + sine": quad_prediction,
               "One box + sine": box["prediction"],
               "Richer carbon model + sine": richer["prediction"]}
scores = pd.DataFrame([error_row(name, pred) for name, pred in predictions.items()]).set_index("Model")
display(scores.round(2))

fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ax.plot(T, C, color="0.6", lw=1, label="Observations")
for (name, pred), color in zip(predictions.items(), ["#2673A1", "#B54C2F", "#27816A"]):
    ax.plot(T, pred, lw=1, color=color, label=name)
ax.axvspan(2017, 2025, color="0.92", zorder=-1)
ax.set(xlabel="Year", ylabel="CO₂ (ppm)", title="More detailed dynamics improve this fit")
ax.legend(fontsize=9)
plt.show()

### Repeat exactly the same emissions experiments

Each future trajectory continues its model's own historical state; we do not reset either model to a later concentration observation. Keep the same vertical scale in both panels.

<a id="predict-richer-response"></a>

**Guess before running:** After emissions become zero, will the richer model lose excess CO₂ faster or slower than the one-box model over the following decades? Explain your guess using the processes represented by each model.

**Answer:** [Your answer here.]

In [ ]:
richer_future = {}
for name, rates in pathways.items():
    edges, trend = four_box_trend(all_years, np.r_[e_all, rates], richer["parameters"])
    richer_future[name] = np.interp(future_times, edges, trend)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True, layout="constrained")
for ax, responses, title in zip(axes, [box_future, richer_future],
                                ["One fixed uptake timescale", "Richer carbon response"]):
    for (name, values), color in zip(responses.items(), colors):
        ax.plot(future_times, values, color=color, label=name)
    ax.plot(future_times, quad_future, color="0.5", ls="--", label="Quadratic continuation")
    ax.set(xlabel="Year", title=title)
    ax.legend(fontsize=8)
axes[0].set_ylabel("CO₂ trend (ppm)")
plt.show()

<a id="model-memory"></a>

**Discuss:** After emissions stop, why does the richer model retain more CO₂? Does its improved historical fit establish that its future pathways are correct?

**Answer:** [Your answer here.]

## Optional experiment: choose your own future emissions

Change the emissions multiplier or the year emissions reach zero. Predict the effect before running. The fitted historical parameters stay fixed.

In [ ]:
EMISSIONS_MULTIPLIER = 1.0
ZERO_YEAR = 2050  # Try 2035, 2050, or 2075; must be later than 2025.
assert ZERO_YEAR > 2025 and EMISSIONS_MULTIPLIER >= 0
chosen_rates = (EMISSIONS_MULTIPLIER*last_rate
                * np.clip((ZERO_YEAR-future_years)/(ZERO_YEAR-2025), 0, 1))
chosen_box = one_box_trend(future_times, box, recent_years, np.r_[e, chosen_rates])
edges, chosen_trend = four_box_trend(all_years, np.r_[e_all, chosen_rates], richer["parameters"])
chosen_richer = np.interp(future_times, edges, chosen_trend)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
axes[0].step(future_years, chosen_rates, where="post", color="0.3")
axes[0].set(xlabel="Year", ylabel="Emissions (ppm/year)", title="Your prescribed pathway")
axes[1].plot(future_times, chosen_box, label="One box", color="#B54C2F")
axes[1].plot(future_times, chosen_richer, label="Richer model", color="#27816A")
axes[1].plot(future_times, quad_future, ls="--", color="0.5", label="Quadratic continuation")
axes[1].set(xlabel="Year", ylabel="CO₂ trend (ppm)", title="Parameters held fixed")
axes[1].legend(fontsize=9)
plt.show()

## What can we learn from the comparison?

Quadratic-plus-sine provides a strong empirical baseline for this record. Dynamical models provide an explicit way to change emissions and follow a state through time. Their responses depend on the equations, initial conditions, and supplied inputs.

The one-box mismatch is part of this example: adding a dynamical equation does not guarantee accuracy. The richer model fits much better, though the quadratic still has the smaller later-period error. Future pathways remain illustrative experiments. We keep the seasonal cycle fixed and approximate a station record using global emissions.

## Sources

- Concentration data: [Scripps CO₂ Program](https://scrippsco2.ucsd.edu/data/atmospheric_co2/). Exact source version and processing are in `data/co2_provenance.json`.
- Emissions: [Global Carbon Budget 2025](https://globalcarbonbudget.org/datahub/the-latest-gcb-data-2025/). Source workbook and accounting are recorded in `data/emissions_provenance.json`. The input uses fossil plus land-use emissions minus the separately reported cement-carbonation sink.
- Units and budget definitions: [Global Carbon Budget 2024 paper](https://essd.copernicus.org/articles/17/965/2025/).
- Carbon responses: [Joos et al. (2013)](https://doi.org/10.5194/acp-13-2793-2013).
- FaIR formulation: [Smith et al. (2018), sections 2.1.1 and 4.2](https://gmd.copernicus.org/articles/11/2273/2018/). Our adaptation uses the published response structure and base timescales, refits uptake coefficients to station observations, and omits temperature feedback.
- Seasonal curve fitting: [NOAA GML](https://gml.noaa.gov/ccgg/mbl/crvfit/).